# 🎨 PixMoji-Diffusion Training Notebook

## Text-to-Pixel Art Generator using Diffusion Transformer (DiT)

This notebook provides a complete training pipeline for PixMoji-Diffusion on Google Colab.

---

## 1️⃣ Environment Setup

In [ ]:
# Check GPU availability
import subprocess
import os

result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,utilization.gpu', '--format=csv,noheader,nounits'],
                       capture_output=True, text=True)
print("🖥️  GPU Information:")
print(result.stdout.strip())

# Set environment variables for better performance
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:512'

# Verify CUDA
import torch
print(f"\n✅ CUDA Available: {torch.cuda.is_available()}")
print(f"   CUDA Version: {torch.version.cuda if torch.cuda.is_available() else 'N/A'}")
print(f"   Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

In [ ]:
# Clone repository and install dependencies
import subprocess

print("📦 Cloning repository...")
subprocess.run(['git', 'clone', 'https://github.com/junyeong-nero/text-to-emoji.git'], check=True)

os.chdir('text-to-emoji')
print(f"📁 Working directory: {os.getcwd()}")

print("\n🔧 Installing dependencies...")
subprocess.run(['uv', 'sync'], check=True)

print("\n✅ Setup complete!")

## 2️⃣ Configuration

In [ ]:
# @title Training Parameters
# @markdown Configure your training parameters below

EPOCHS = 100  # @param {type:"integer"}
BATCH_SIZE = 64  # @param {type:"integer"}
LEARNING_RATE = 3e-4  # @param {type:"number"}
MODEL_SIZE = "S"  # @param ["S", "B", "L"]
USE_WANDB = False  # @param {type:"boolean"}

print("=" * 50)
print("🎯 Training Configuration")
print("=" * 50)
print(f"   Epochs:         {EPOCHS}")
print(f"   Batch Size:     {BATCH_SIZE}")
print(f"   Learning Rate:  {LEARNING_RATE}")
print(f"   Model Size:     {MODEL_SIZE}")
print(f"   W&B Enabled:    {USE_WANDB}")
print("=" * 50)

## 3️⃣ Training

In [ ]:
# Build and execute training command
import subprocess

cmd = [
    "uv", "run", "python", "src/training/train.py",
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--learning-rate", str(LEARNING_RATE),
    "--model-size", MODEL_SIZE,
]

if USE_WANDB:
    cmd.append("--wandb")

print("🚀 Starting Training...")
print(f"Command: {' '.join(cmd)}")
print()

# Run training
result = subprocess.run(cmd)

if result.returncode == 0:
    print("\n✅ Training completed successfully!")
else:
    print(f"\n❌ Training failed with return code: {result.returncode}")

## 4️⃣ Inference - Generate Emojis

In [ ]:
# Generate emojis from text prompts
import subprocess
import os

# Create output directory
os.makedirs('generated', exist_ok=True)

# Prompts to generate
prompts = [
    "a cute robot",
    "a smiling face with hearts",
    "a rocket ship launching",
    "a ghost wearing a hat",
    "a red apple",
    "an astronaut in space",
]

print("🎨 Generating Emojis")
print("=" * 50)

for prompt in prompts:
    print(f"\n🔮 Prompt: '{prompt}'")
    
    cmd = [
        "uv", "run", "python", "src/inference/generate.py",
        "--prompt", prompt,
        "--num-samples", "2",
        "--checkpoint", "checkpoints/model_final.pt",
        "--guidance-scale", "7.5",
        "--steps", "50",
    ]
    
    subprocess.run(cmd, capture_output=True)

print("\n" + "=" * 50)
print("✅ Generation complete! Check 'generated' folder.")

## 5️⃣ Display Generated Images

In [ ]:
# Display generated images
from PIL import Image
import matplotlib.pyplot as plt
import os
import glob

def display_generated_images(output_dir='generated', num_images=8):
    """Display generated emoji images."""
    
    # Find all generated images
    image_files = glob.glob(os.path.join(output_dir, '*.png'))
    
    if not image_files:
        print("No generated images found in", output_dir)
        return
    
    # Sort by modification time
    image_files.sort(key=os.path.getmtime, reverse=True)
    
    # Display up to num_images
    num_images = min(num_images, len(image_files))
    
    fig, axes = plt.subplots(1, num_images, figsize=(4 * num_images, 4))
    if num_images == 1:
        axes = [axes]
    
    for i, img_path in enumerate(image_files[:num_images]):
        img = Image.open(img_path)
        axes[i].imshow(img)
        axes[i].axis('off')
        axes[i].set_title(os.path.splitext(os.path.basename(img_path))[0], fontsize=10)
    
    plt.tight_layout()
    plt.show()

# Display generated images
print("🖼️  Generated Emojis:")
display_generated_images()

## 6️⃣ Download Checkpoints

In [ ]:
# Download trained model to local machine
from google.colab import files
import os

print("📦 Available Checkpoints:")
print("-" * 40)

checkpoint_dir = 'checkpoints'
if os.path.exists(checkpoint_dir):
    for f in os.listdir(checkpoint_dir):
        fpath = os.path.join(checkpoint_dir, f)
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f"   {f} ({size_mb:.1f} MB)")

print("-" * 40)
print("\n⬇️  Download checkpoints to your local machine:")
files.download(checkpoint_dir + '/model_final.pt')

## 📚 Additional Resources

In [ ]:
# View training logs
import subprocess

# List recent files
print("📁 Recent Files:")
subprocess.run(['ls', '-lh', 'checkpoints'], capture_output=True)

# Check W&B (if enabled)
if USE_WANDB:
    print("\n📊 View your W&B run at: https://wandb.ai")
else:
    print("\n💡 Enable W&B by setting USE_WANDB = True for experiment tracking")